Import the required libraries

In [4]:
import requests
import io, gzip, zipfile
import os
import snowflake.connector
import re

Download files from BTS source using through requests

In [ ]:
for year in range(2025, 2027):
    for month in range(1,13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        open(f"ontime_{year}_{month}.zip", "wb").write(r.content)

Create the connection to snowflake and put the credetials in a separate file.
Also created a warehouse in snowflake, a database and a schema to store the data.

In [7]:
conn = snowflake.connector.connect(
    user=os.environ.get('SNOWFLAKE_USER'),
    password=os.environ.get('SNOWFLAKE_PASSWORD'),
    account=os.environ.get('SNOWFLAKE_ACCOUNT'),
    warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE'),
    database=os.environ.get('SNOWFLAKE_DATABASE'),
    schema=os.environ.get('SNOWFLAKE_SCHEMA')
)

Before the next step we need to create a SCHEMA and STAGES in snowflake. Let's create one STAGE for BTS data that we downloaded and one for NOAA Weather data which is a connection to a S3 bucket.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE SCHEMA IF NOT EXISTS RAW_DATA;
    """)

with conn.cursor() as cur:
    cur.execute("""
        CREATE STAGE IF NOT EXISTS RAW_DATA.NOAA_GHCN_STAGE
          URL = 's3://noaa-ghcn-pds/'
          FILE_FORMAT = (TYPE = CSV)
    """)

with conn.cursor() as cur:
    cur.execute("""
        CREATE STAGE IF NOT EXISTS RAW_DATA.BTS_STAGE
    """)

Unzip the downloaded files locally and push them to the snowflake staging area called @staging_data. 

In [ ]:
for file in os.listdir():
    print(f"Found {file}")
    if file.startswith("ontime_") and file.endswith(".zip"):
        print(f"Processing {file} ...")
        with open(file, "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        # clean output name: no parentheses, ends in .csv 
                        out = re.sub(r"[()]", "", os.path.basename(name)) + ".gz"
                        with z.open(name) as csv_file, gzip.open(out, "wt", encoding="utf-8", newline="") as dst:
                            for line in io.TextIOWrapper(csv_file, encoding="utf-8", errors="replace", newline=""):
                                line = line.rstrip("\r\n")
                                if line.endswith(","):
                                    line = line[:-1]          # drop only the one trailing comma
                                dst.write(line + "\n")
                        print(f"Uploading {out}")
                        conn.cursor().execute(f"PUT 'file://{out}' @RAW_DATA.BTS_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

So after we downloaded all the files there is an extra step if we are using a trial account. We ideally want to upload this files to snowflake. If not using a trial account, we can directly run this notebook inside snowflake, perhaps it needs some folder creation and changes to code.

After we successfully moved the files to snowflake, now we need to remove them locally.

In [ ]:
for file in os.listdir():
    if file.endswith("gz") or file.endswith("zip"):
        print(f"Removing {file} ...")
        os.remove(file)

Next step is to create the weather tables for precipitation, temperatures, and wind
But first I am going to clone the dbt-snowflake getting started project to this repo and link github with snowflake account.

Lets download carrier lookup data to know exactly what carrier is having most delays.

In [5]:
url = 'https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef'
r = requests.get(url, timeout=120)
r.raise_for_status()
open("data/bts/carrier_lookup.csv", "wb").write(r.content)

54240

In [ ]:
file_name = "data/bts/carrier_lookup.csv"
print(f"Uploading {file_name}")
conn.cursor().execute(f"PUT 'file://{file_name}' @RAW_DATA.BTS_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

Uploading data/bts/carrier_lookup.csv


After downloading the carrier lookup, now we need the airport lookup. So we need to run the download_t_master_cord_selenium.py, we need this lookup data for airport coordinates. I'm adding a separate python execution cause the execution of selenium in this jupyter notebook is crashing python, probably something related to chrome driver. So just run 'python download_airport_cord_selenium.py' and it will download and extract file to the folder

Donwload lookup airport data from BTS using selenium

In [ ]:
!python download_airport_cord_selenium.py

[info] 33 field checkboxes selected
[info] download clicked, waiting for the file...
[ok] saved /Users/amandacostaelias/code/flight_weather_project/data/bts/AIRPORT_CORD.csv (from T_MASTER_CORD.csv)


Now Lets send this file to staging_data in snowflake

In [ ]:
conn.cursor().execute(f"PUT 'file://data/bts/AIRPORT_CORD.csv' @RAW_DATA.BTS_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

After all the data is downloaded and stored inside the STAGES, lets create the RAW tables